In [1]:
import torch

from floodvision.models.unet import UNet

model = UNet(
    in_channels=2,
    out_channels=1,
    base_channels=16
)

print(model)

UNet(
  (encoder1): DoubleConv(
    (block): Sequential(
      (0): Conv2d(2, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (encoder2): DoubleConv(
    (block): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (5): ReLU(inplace=True)

In [2]:
from floodvision.models.baseline_cnn import BaselineCNN

baseline = BaselineCNN(
    in_channels=2,
    out_channels=1,
    base_channels=16
)

baseline_params = sum(
    p.numel() for p in baseline.parameters()
)

unet_params = sum(
    p.numel() for p in model.parameters()
)

additional_params = unet_params - baseline_params

increase = (additional_params / baseline_params) * 100

print(f"Baseline CNN parameters: {baseline_params:,}")
print(f"U-Net parameters: {unet_params:,}")
print(f"Additional parameters: {additional_params:,}")
print(f"Parameter increase: {increase:.2f}%")

Baseline CNN parameters: 434,209
U-Net parameters: 482,593
Additional parameters: 48,384
Parameter increase: 11.14%


In [3]:
model.eval()

x = torch.randn(1, 2, 512, 512)

with torch.inference_mode():
    output = model(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Output dtype:", output.dtype)
print("Output min:", output.min().item())
print("Output max:", output.max().item())

assert output.shape == (1, 1, 512, 512)
assert torch.isfinite(output).all()

print("U-Net forward pass successful!")

Input shape: torch.Size([1, 2, 512, 512])
Output shape: torch.Size([1, 1, 512, 512])
Output dtype: torch.float32
Output min: -0.049228325486183167
Output max: 0.012076986953616142
U-Net forward pass successful!


In [4]:
from floodvision.losses.combined import BCEDiceLoss

model.train()
model.zero_grad(set_to_none=True)

# Synthetic input
x = torch.randn(2, 2, 128, 128)

# Synthetic segmentation mask
target = torch.randint(
    0, 2, (2, 1, 128, 128)
).float()

# Simulate invalid pixels
target[:, :, :10, :10] = -1

# Actual project loss
criterion = BCEDiceLoss(
    bce_weight=0.5,
    dice_weight=0.5,
    pos_weight=None
)

# Forward pass
logits = model(x)

# Calculate loss
loss = criterion(logits, target)

# Backward pass
loss.backward()

print("Loss:", loss.item())

missing_gradients = []
invalid_gradients = []
nonzero_gradients = 0

for name, param in model.named_parameters():

    if not param.requires_grad:
        continue

    if param.grad is None:
        missing_gradients.append(name)

    elif not torch.isfinite(param.grad).all():
        invalid_gradients.append(name)

    elif torch.count_nonzero(param.grad).item() > 0:
        nonzero_gradients += 1

print("Missing gradients:", missing_gradients)
print("Invalid gradients:", invalid_gradients)
print("Parameter tensors with nonzero gradients:", nonzero_gradients)

assert torch.isfinite(loss)
assert not missing_gradients
assert not invalid_gradients
assert nonzero_gradients > 0

print("U-Net backward pass successful!")

Loss: 0.6432074308395386
Missing gradients: []
Invalid gradients: []
Parameter tensors with nonzero gradients: 50
U-Net backward pass successful!


In [5]:
from floodvision.models.factory import build_model

model_config = {
    "name": "unet",
    "in_channels": 2,
    "out_channels": 1,
    "base_channels": 16
}

factory_model = build_model(model_config)

print("Model class:", factory_model.__class__.__name__)

print(
    "Total parameters:",
    f"{sum(p.numel() for p in factory_model.parameters()):,}"
)

assert isinstance(factory_model, UNet)

print("Model factory validation passed!")

Model class: UNet
Total parameters: 482,593
Model factory validation passed!
